In [1]:
# ============================================================
# 3D Fusion (Tokamak) — MHD-Inspired Sandbox + URT Controller
# ============================================================
# What you get:
# - 3D torus grid (r, θ, φ) and a scalar plasma field P(x,y,z)
# - Advection (shear flow), diffusion, nonlinear saturation
# - Low-order unstable harmonics (1/1, 2/1, 2/2) with wall-like growth
# - 4 external coils you can actuate (u ∈ R^4) mapped to harmonics
# - Fast URT controller (stable if κ = β α (1+θ_h) < 1; auto-clamped)
# - Metrics + (optional) Plotly isosurface; otherwise Matplotlib fallback
# NOTE: Didactic sandbox (NOT a physical MHD solver).
# ------------------------------------------------------------

import numpy as np
import math, time, warnings

# Optional Plotly for 3D isosurface
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (import side-effects)

# ------------------------ Utils ------------------------
def safe_norm(x, eps=1e-12):
    return float(np.sqrt(np.sum(x*x) + eps))

def roll3(A, si=0, sj=0, sk=0):
    if si: A = np.roll(A, si, axis=0)
    if sj: A = np.roll(A, sj, axis=1)
    if sk: A = np.roll(A, sk, axis=2)
    return A

# --------------------- URT Controller -------------------
class FastURT:
    """
    u_{k+1} = β * [ α( u_k − θ_h*φ(u_k) ) + v_k ],
    where v_k is a control "learning" signal (here from mode amplitudes).
    φ is smooth odd; we use tanh for saturation & nice gradients.

    Auto-stability: if κ=β α (1+θ_h) ≥ κ_max, β is clamped so κ<κ_max.
    inner_loops: do several internal URT iterations per outer step for extra contraction.
    """
    def __init__(self, alpha=1.0, theta_h=2.0, beta=0.30, kappa_max=0.95, inner_loops=1):
        self.alpha = float(alpha)
        self.theta_h = float(theta_h)
        self.beta = float(beta)
        self.inner_loops = int(max(1, inner_loops))
        # Clamp beta if needed
        kappa = self.beta * self.alpha * (1.0 + self.theta_h)
        if kappa >= kappa_max:
            self.beta = (kappa_max - 1e-6) / (self.alpha * (1.0 + self.theta_h))
            kappa = self.beta * self.alpha * (1.0 + self.theta_h)
        print(f"Stability verified: κ={kappa:.3f} (β={self.beta:.3f}, inner_loops={self.inner_loops})")

    def phi(self, x):
        return np.tanh(x)

    def _one(self, u, v):
        return self.beta * ( self.alpha * (u - self.theta_h * self.phi(u)) + v )

    def step(self, u, v):
        for _ in range(self.inner_loops):
            u = self._one(u, v)
        return u

# --------------- 3D Tokamak MHD-inspired model ---------------
class Tokamak3D_MHD:
    """
    3D torus on structured grid (r, θ, φ) with pressure-like scalar P.
    Very simple advection-diffusion + nonlinear saturation and
    low-mode edge harmonics that grow unless controlled.
    4 coils influence the field; they’re mapped to harmonics.

    Grid: Nr×Nth×Nph (axes: r, θ, φ)
    Coordinates -> Cartesian for visualization.
    """
    def __init__(self,
                 R0=1.7, a=0.5, Nr=24, Nth=48, Nph=48, seed=0,
                 nu=2.0e-3, dt=0.8, Psat=1.2,
                 v_pol0=0.15, v_tor0=0.25, shear=0.6,
                 wall_gamma={ (1,1): 0.012, (2,1): 0.007, (2,2): 0.004 },
                 coil_gain=0.08, noise_sigma=2e-4, disruption_p=0.0):
        rng = np.random.default_rng(seed)
        self.R0, self.a = float(R0), float(a)
        self.Nr, self.Nth, self.Nph = int(Nr), int(Nth), int(Nph)

        # Grid
        r  = np.linspace(0, a, Nr)
        th = np.linspace(0, 2*np.pi, Nth, endpoint=False)
        ph = np.linspace(0, 2*np.pi, Nph, endpoint=False)
        R, TH, PH = np.meshgrid(r, th, ph, indexing='ij')
        self.R, self.TH, self.PH = R, TH, PH

        # Cartesian for plotting
        X = (R0 + R*np.cos(TH)) * np.cos(PH)
        Y = (R0 + R*np.cos(TH)) * np.sin(PH)
        Z = R * np.sin(TH)
        self.X, self.Y, self.Z = X, Y, Z

        # Base equilibrium
        base = np.exp(-(R**2) / (0.8*a)**2)
        self.P = base.copy()

        # Dynamics params
        self.nu = float(nu)
        self.dt = float(dt)
        self.Psat = float(Psat)
        self.v_pol0 = float(v_pol0)
        self.v_tor0 = float(v_tor0)
        self.shear = float(shear)
        self.noise_sigma = float(noise_sigma)
        self.disruption_p = float(disruption_p)
        self.coil_gain = float(coil_gain)

        # Modes at the edge
        self.modes = [(1,1), (2,1), (2,2)]
        self.wall_gamma = dict(wall_gamma)
        self.Amn = {k: 1e-3 for k in self.modes}
        self.phi0 = {k: rng.uniform(0, 2*np.pi) for k in self.modes}

        # Coils: 4 coils at φ=0, π/2, π, 3π/2, θ=0
        self.num_coils = 4
        self.coil_phi = np.array([0, 0.5*np.pi, np.pi, 1.5*np.pi])
        self.coil_th  = np.zeros_like(self.coil_phi)
        self.coil_R   = R0 + a + 0.05
        self.u = np.zeros(self.num_coils)  # coil currents

        self.coil_fields = self._build_coil_fields()

        # Diagnostics history
        self.history = { "t": [], "A11": [], "A21": [], "A22": [], "u_norm": [], "P_std": [] }

    # ---- internals ----
    def _build_coil_fields(self):
        fields = []
        for phi_c in self.coil_phi:
            xc = (self.coil_R) * np.cos(phi_c)
            yc = (self.coil_R) * np.sin(phi_c)
            zc = 0.0
            dx, dy, dz = self.X - xc, self.Y - yc, self.Z - zc
            dist2 = dx*dx + dy*dy + dz*dz
            f = 1.0 / (dist2 + 1e-3)
            f /= np.max(f)
            fields.append(f)
        return np.array(fields)  # (4, Nr, Nth, Nph)

    def _harmonic_basis(self, m, n):
        # Edge-weighted cosine
        edge = np.clip(self.R / max(self.a, 1e-9), 0, 1)
        return edge * np.cos(m*self.TH + n*self.PH + self.phi0[(m,n)])

    def _measure_modes(self):
        amps = {}
        w = (self.R > 0.6*self.a).astype(float)
        denom_cache = {}
        for (m,n) in self.modes:
            B = self._harmonic_basis(m,n)
            key = (m,n)
            if key not in denom_cache:
                denom_cache[key] = np.sum(B*B*w) + 1e-12
            amps[key] = float(np.sum(self.P*B*w) / denom_cache[key])
        return amps

    def _coil_to_harmonics_map(self):
        # Map u(4) -> ΔA(3)
        M = []
        for (m,n) in self.modes:
            B = self._harmonic_basis(m,n)
            row = [np.sum(self.coil_fields[i] * B) for i in range(self.num_coils)]
            M.append(row)
        M = np.array(M)
        M /= (np.max(np.abs(M)) + 1e-9)
        return M  # (3, 4)

    def _advect_diffuse(self, P):
        # Simple centered advection in θ, φ with shear; Laplacian diffusion in θ, φ, r
        # θ/φ steps (assume unit)
        dθ = 1.0; dφ = 1.0
        # r step (uniform)
        dr = self.a / max(1, (self.Nr - 1))

        # shear profile: more at the edge
        s = (self.R / max(self.a, 1e-9))**self.shear
        vθ = self.v_pol0 * (0.3 + 0.7*s)    # poloidal
        vφ = self.v_tor0 * (0.5 + 0.5*s)    # toroidal

        # centered differences (periodic in θ, φ)
        dP_dθ = (roll3(P, sj=+1) - roll3(P, sj=-1)) / (2*dθ)
        dP_dφ = (roll3(P, sk=+1) - roll3(P, sk=-1)) / (2*dφ)
        adv = -(vθ * dP_dθ + vφ * dP_dφ)

        # diffusion Laplacian (θ, φ)
        lap_th = roll3(P, sj=+1) - 2.0*P + roll3(P, sj=-1)
        lap_ph = roll3(P, sk=+1) - 2.0*P + roll3(P, sk=-1)

        # crude radial diffusion (Neumann at r=0,a)
        Prp = np.copy(P); Prm = np.copy(P)
        Prp[:-1] = P[1:]; Prm[1:] = P[:-1]
        lap_r = (Prp - 2.0*P + Prm)

        diff = self.nu * (lap_th + lap_ph + 0.3*lap_r)  # small radial factor

        return P + self.dt * (adv + diff)

    def step(self, growth=True, control=True):
        # (1) Start from equilibrium + current modes
        base = np.exp(-(self.R**2) / (0.8*self.a)**2)
        P = base.copy()
        for (m,n), A in self.Amn.items():
            P += A * self._harmonic_basis(m,n)

        # (2) Coil influence
        if control:
            coil_field = np.tensordot(self.u, self.coil_fields, axes=(0,0))
            P += self.coil_gain * coil_field

        # (3) Advection + diffusion
        P = self._advect_diffuse(P)

        # (4) Nonlinear saturation (soft quench)
        # P <- P / (1 + (P/Psat)^2)
        P = P / (1.0 + (P / self.Psat)**2)

        # (5) Noise & rare "disruption" blips (toy)
        if self.noise_sigma > 0:
            P += np.random.normal(0.0, self.noise_sigma, size=P.shape)
        if self.disruption_p > 0 and np.random.rand() < self.disruption_p:
            bump = np.exp(-((self.R - 0.8*self.a)**2) / (0.03*self.a)**2)
            P += 0.15 * bump * np.cos(self.TH - self.PH)

        # (6) Measure modes and update their amplitudes with wall-like growth
        amps = self._measure_modes()
        if growth:
            for key in self.modes:
                g = self.wall_gamma[key]
                self.Amn[key] = float(amps[key]) * (1.0 + g)

        # (7) Save
        self.P = P
        self.history["t"].append(len(self.history["t"]))
        self.history["A11"].append(amps[(1,1)])
        self.history["A21"].append(amps[(2,1)])
        self.history["A22"].append(amps[(2,2)])
        self.history["u_norm"].append(np.linalg.norm(self.u))
        self.history["P_std"].append(float(np.std(P)))

        return amps

# ----------------- Closed-loop run harness -----------------
def run_mhd_fusion(steps=120,
                   urt_alpha=1.0, urt_theta_h=2.0, urt_beta=0.30,
                   urt_kappa_max=0.95, urt_inner_loops=5,
                   R0=1.7, a=0.5, Nr=24, Nth=48, Nph=48,
                   seed=1, objective_weights=(1.0, 0.7, 0.5),
                   growth=True, control=True, show_progress=True):
    print("Initializing model + URT …")
    urt = FastURT(alpha=urt_alpha, theta_h=urt_theta_h, beta=urt_beta,
                  kappa_max=urt_kappa_max, inner_loops=urt_inner_loops)
    sim = Tokamak3D_MHD(R0=R0, a=a, Nr=Nr, Nth=Nth, Nph=Nph, seed=seed)

    # Coil mapping (3 harmonics ← 4 coils)
    M = sim._coil_to_harmonics_map()  # (3,4)
    Minv = np.linalg.pinv(M)          # (4,3)
    w = np.array(objective_weights, dtype=float).reshape(3)

    for k in range(steps):
        amps = sim.step(growth=growth, control=True)
        a_vec = np.array([amps[(1,1)], amps[(2,1)], amps[(2,2)]])
        # Desired delta on harmonics: negative of their measured values (weighted)
        v = Minv @ (-w * a_vec)
        sim.u = urt.step(sim.u, v)
        if show_progress and ((k+1) % max(1, steps//6) == 0 or k == 0):
            print(f" step {k+1:>3d} | A11={a_vec[0]: .3e}  A21={a_vec[1]: .3e}  A22={a_vec[2]: .3e} | ‖u‖={np.linalg.norm(sim.u):.3f}")

    # Final amplitudes
    amps = sim._measure_modes()
    print("\nFinal amplitudes:")
    for key in sim.modes:
        print(f"  A{key} = {amps[key]: .3e}")
    return sim

# -------------------- Visualization --------------------
def plot_metrics(sim: Tokamak3D_MHD, title_suffix="(URT closed-loop)"):
    t = np.array(sim.history["t"], dtype=float)
    A11 = np.abs(np.array(sim.history["A11"]))
    A21 = np.abs(np.array(sim.history["A21"]))
    A22 = np.abs(np.array(sim.history["A22"]))
    uN  = np.array(sim.history["u_norm"])
    Psd = np.array(sim.history["P_std"])

    fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    ax[0].plot(t, A11, label="|A(1,1)|")
    ax[0].plot(t, A21, label="|A(2,1)|")
    ax[0].plot(t, A22, label="|A(2,2)|")
    ax[0].set_yscale("log")
    ax[0].set_ylabel("Mode amplitude (log)")
    ax[0].grid(True, alpha=0.3); ax[0].legend()

    ax[1].plot(t, uN, label="‖u‖")
    ax[1].set_ylabel("Control norm")
    ax[1].grid(True, alpha=0.3); ax[1].legend()

    ax[2].plot(t, Psd, label="std(P)")
    ax[2].set_xlabel("Step"); ax[2].set_ylabel("Field std")
    ax[2].grid(True, alpha=0.3); ax[2].legend()

    fig.suptitle(f"Tokamak Sandbox Metrics {title_suffix}")
    plt.tight_layout(); plt.show()

def show_isosurface(sim: Tokamak3D_MHD, iso=0.40):
    P = sim.P.copy()
    Pn = (P - P.min()) / (P.max() - P.min() + 1e-12)

    if HAS_PLOTLY:
        print("Rendering Plotly isosurface …")
        fig = go.Figure(data=go.Isosurface(
            x=sim.X.flatten(), y=sim.Y.flatten(), z=sim.Z.flatten(),
            value=Pn.flatten(),
            isomin=iso, isomax=iso,
            surface_count=1,
            caps=dict(x_show=False, y_show=False, z_show=False),
        ))
        fig.update_layout(
            title=f"3D Plasma Isosurface (iso={iso:.2f})",
            scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
            width=900, height=700
        )
        fig.show()
    else:
        print("Plotly not found. Falling back to Matplotlib 3D scatter …")
        mask = Pn > iso
        x, y, z = sim.X[mask], sim.Y[mask], sim.Z[mask]
        # subsample if too many points
        if x.size > 40000:
            idx = np.random.choice(x.size, 40000, replace=False)
            x, y, z = x.flatten()[idx], y.flatten()[idx], z.flatten()[idx]
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(x, y, z, s=0.8, alpha=0.5)
        ax.set_title(f"3D Plasma Isosurface (iso={iso:.2f})")
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.view_init(22, 35)
        plt.show()

# -------------------- Open-loop baseline --------------------
def run_open_loop_baseline(steps=120, **sim_kwargs):
    sim = Tokamak3D_MHD(**sim_kwargs)
    for _ in range(steps):
        sim.step(growth=True, control=False)
    return sim

# ========================== RUN DEMO ==========================
if __name__ == "__main__":
    params = dict(
        steps=120,
        urt_alpha=1.0,
        urt_theta_h=2.0,
        urt_beta=0.30,        # κ_base ≈ 0.90; auto-clamped if needed
        urt_kappa_max=0.95,   # clamp κ below this
        urt_inner_loops=8,    # more inner loops -> stronger contraction per outer step
        R0=1.7, a=0.5,
        Nr=22, Nth=44, Nph=44,   # modest grid so Colab runs smoothly
        seed=2,
        objective_weights=(1.0, 0.7, 0.5),
        growth=True, control=True, show_progress=True
    )

    sim_cl = run_mhd_fusion(**params)
    plot_metrics(sim_cl, title_suffix="(URT on)")
    show_isosurface(sim_cl, iso=0.42)

    print("\nRunning open-loop baseline …")
    sim_ol = run_open_loop_baseline(
        steps=params["steps"], R0=params["R0"], a=params["a"],
        Nr=params["Nr"], Nth=params["Nth"], Nph=params["Nph"], seed=params["seed"]
    )
    plot_metrics(sim_ol, title_suffix="(open loop)")
    show_isosurface(sim_ol, iso=0.42)

    # quick summary
    def summarize(sim, name):
        amps = sim._measure_modes()
        s = f"{name}: " + ", ".join([f"A{mn}={amps[mn]:.3e}" for mn in sim.modes])
        print(s)
    summarize(sim_cl, "Closed loop (URT)")
    summarize(sim_ol, "Open loop     ")

Output hidden; open in https://colab.research.google.com to view.